In [ ]:
import numpy as np
import cv2
import os
import tensorflow as tf
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.model_selection import train_test_split
from sklearn.covariance import EmpiricalCovariance
from sklearn.metrics import classification_report, confusion_matrix
from scipy.special import logsumexp
from skimage.feature import local_binary_pattern
from skimage.filters import gabor
import pywt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from scipy.stats import skew, kurtosis

In [ ]:
IMG_SIZE     = 128
BATCH_SIZE   = 16
EPOCHS       = 30
MC_SAMPLES   = 20
RANDOM_SEED  = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

print("Configuration set!")

Configuration set!


In [ ]:
import kagglehub
path = kagglehub.dataset_download(
    "ahsanneural/rare-neurological-diseases-mri-curated-edition"
)
print("Dataset path:", path)
DATASET_PATH = path

Using Colab cache for faster access to the 'rare-neurological-diseases-mri-curated-edition' dataset.
Dataset path: /kaggle/input/rare-neurological-diseases-mri-curated-edition


In [ ]:
disease_classes_set = set()

for top_level in os.listdir(DATASET_PATH):
    top_path = os.path.join(DATASET_PATH, top_level)
    if not os.path.isdir(top_path):
        continue
    for sub in os.listdir(top_path):
        sub_path = os.path.join(top_path, sub)
        if not os.path.isdir(sub_path):
            continue
        for cls in os.listdir(sub_path):
            if os.path.isdir(os.path.join(sub_path, cls)):
                disease_classes_set.add(cls)

classes       = sorted(list(disease_classes_set))
n_classes     = len(classes)
class_to_label = {name: i for i, name in enumerate(classes)}

print(f"Discovered {n_classes} classes: {classes}")

Discovered 5 classes: ['fukuyama_muscular_dystrophy', 'hallervorden_spatz_disease', 'moyamoya_disease', 'pachygyria_cerebellar_hypoplasia', 'walker_warburg_syndrome']


In [ ]:
counts = []
for c in classes:
    count = 0
    for top in os.listdir(DATASET_PATH):
        tp = os.path.join(DATASET_PATH, top)
        if not os.path.isdir(tp):
            continue
        for sub in os.listdir(tp):
            sp = os.path.join(tp, sub)
            dp = os.path.join(sp, c)
            if os.path.isdir(dp):
                count += len([f for f in os.listdir(dp)
                               if f.lower().endswith(('.png','.jpg','.jpeg'))])
    counts.append(count)

fig, ax = plt.subplots(figsize=(12, 5))
colors  = plt.cm.tab20(np.linspace(0, 1, n_classes))
bars    = ax.bar(classes, counts, color=colors, edgecolor='black', linewidth=0.5)
ax.set_title("Class Distribution – Rare Neurological Diseases MRI Dataset",
             fontsize=13, fontweight='bold')
ax.set_xlabel("Disease Class", fontsize=11)
ax.set_ylabel("Number of Images", fontsize=11)
ax.tick_params(axis='x', rotation=45)
for bar, cnt in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 5,
            str(cnt), ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig('/content/fig1_class_distribution.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: fig1_class_distribution.png")

Saved: fig1_class_distribution.png


In [ ]:
print("Loading dataset...")
X_list, y_list = [], []
skipped = 0

for split in os.listdir(DATASET_PATH):
    split_path = os.path.join(DATASET_PATH, split)
    if not os.path.isdir(split_path):
        continue
    for sub in os.listdir(split_path):
        sub_path = os.path.join(split_path, sub)
        if not os.path.isdir(sub_path):
            continue
        for cls in os.listdir(sub_path):
            cls_path = os.path.join(sub_path, cls)
            if not os.path.isdir(cls_path) or cls not in class_to_label:
                continue
            for fname in os.listdir(cls_path):
                if not fname.lower().endswith(('.png','.jpg','.jpeg')):
                    continue
                fpath = os.path.join(cls_path, fname)
                try:
                    img = tf.keras.utils.load_img(
                        fpath,
                        target_size=(IMG_SIZE, IMG_SIZE),
                        color_mode='rgb'
                    )
                    img_array = tf.keras.utils.img_to_array(img)

                    if img_array.shape != (IMG_SIZE, IMG_SIZE, 3):
                        skipped += 1
                        continue
                    if np.isnan(img_array).any():
                        skipped += 1
                        continue

                    img_array = img_array / 255.0   # normalize to [0,1]
                    X_list.append(img_array)
                    y_list.append(class_to_label[cls])

                except Exception as e:
                    skipped += 1
                    continue

print(f"Loaded : {len(X_list)} images")
print(f"Skipped: {skipped} images")

X = np.array(X_list, dtype=np.float32)
y = np.array(y_list, dtype=np.int32)

print(f"\nX shape : {X.shape}")
print(f"y shape : {y.shape}")
print(f"X range : {X.min():.3f} → {X.max():.3f}")

print("\nLabel distribution:")
unique, counts_y = np.unique(y, return_counts=True)
for idx, cnt in zip(unique, counts_y):
    print(f"  {classes[idx]:40s}: {cnt} images")

y_cat = tf.keras.utils.to_categorical(y, num_classes=n_classes)
print(f"\ny_cat shape  : {y_cat.shape}")
print(f"Label check  : y[0]={y[0]} → y_cat[0]={y_cat[0]}")

Loading dataset...
Loaded : 2000 images
Skipped: 0 images

X shape : (2000, 128, 128, 3)
y shape : (2000,)
X range : 0.000 → 1.000

Label distribution:
  fukuyama_muscular_dystrophy             : 400 images
  hallervorden_spatz_disease              : 400 images
  moyamoya_disease                        : 400 images
  pachygyria_cerebellar_hypoplasia        : 400 images
  walker_warburg_syndrome                 : 400 images

y_cat shape  : (2000, 5)
Label check  : y[0]=1 → y_cat[0]=[0. 1. 0. 0. 0.]


In [ ]:
def radiomics_features(img: np.ndarray) -> np.ndarray:
    gray    = img[:, :, 0]
    gray_u8 = (gray * 255).astype(np.uint8)

    # LBP
    lbp      = local_binary_pattern(gray, P=8, R=1, method='uniform')
    lbp_hist, _ = np.histogram(lbp.ravel(), bins=26, range=(0, 26), density=True)

    # Gabor
    gabor_feats = []
    for theta in np.linspace(0, np.pi, 8, endpoint=False):
        for freq in [0.1, 0.3, 0.5]:
            filt_r, filt_i = gabor(gray, frequency=freq, theta=theta)
            gabor_feats.append(np.sqrt(filt_r**2 + filt_i**2).mean())
    gabor_feats = np.array(gabor_feats)

    # Wavelet
    wav_feats = []
    for level in [1, 2]:
        coeffs       = pywt.dwt2(gray, 'haar')
        cA, (cH, cV, cD) = coeffs
        for c in [cH, cV, cD, cA]:
            wav_feats.append(np.mean(c**2))
    wav_feats = np.array(wav_feats[:8])

    # Stats
    flat       = gray.ravel()
    stats_feats = np.array([flat.mean(), flat.std(),
                             skew(flat), kurtosis(flat)])

    return np.concatenate([lbp_hist, gabor_feats, wav_feats, stats_feats])

print("Extracting radiomics features...")
R = np.array([radiomics_features(img) for img in X])
print(f"Radiomics shape: {R.shape}")

Extracting radiomics features...
Radiomics shape: (2000, 62)


In [ ]:
# Shuffle before splitting
shuffle_idx = np.random.permutation(len(X))
X     = X[shuffle_idx]
y     = y[shuffle_idx]
y_cat = y_cat[shuffle_idx]
R     = R[shuffle_idx]

X_train, X_test, R_train, R_test, y_train, y_test = train_test_split(
    X, R, y_cat,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=y
)

print(f"Train samples : {len(X_train)}")
print(f"Test samples  : {len(X_test)}")
print(f"Train label distribution: {np.bincount(y_train.argmax(axis=1))}")
print(f"Test  label distribution: {np.bincount(y_test.argmax(axis=1))}")

Train samples : 1600
Test samples  : 400
Train label distribution: [320 320 320 320 320]
Test  label distribution: [80 80 80 80 80]


In [ ]:
# ← LIGHTER augmentation for medical images
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),   # ← horizontal only
    tf.keras.layers.RandomRotation(0.1),        # ← gentle rotation
    tf.keras.layers.RandomZoom(0.1),            # ← gentle zoom
    tf.keras.layers.RandomContrast(0.1),        # ← gentle contrast
], name="augmentation")

print("Augmentation pipeline ready!")

Augmentation pipeline ready!


In [ ]:

def build_cnn(n_classes: int) -> tf.keras.Model:
    inp = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="image_input")


    # Block 1
    x = tf.keras.layers.Conv2D(32, 3, padding='same')(inp)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = tf.keras.layers.Conv2D(32, 3, padding='same')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = tf.keras.layers.MaxPooling2D(2)(x)
    x = tf.keras.layers.Dropout(0.25)(x)

    # Block 2
    x = tf.keras.layers.Conv2D(64, 3, padding='same')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = tf.keras.layers.Conv2D(64, 3, padding='same')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = tf.keras.layers.MaxPooling2D(2)(x)
    x = tf.keras.layers.Dropout(0.25)(x)

    # Block 3
    x = tf.keras.layers.Conv2D(128, 3, padding='same')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = tf.keras.layers.Conv2D(128, 3, padding='same')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = tf.keras.layers.MaxPooling2D(2)(x)
    x = tf.keras.layers.Dropout(0.25)(x)

    # Block 4
    x = tf.keras.layers.Conv2D(256, 3, padding='same')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = tf.keras.layers.MaxPooling2D(2)(x)
    x = tf.keras.layers.Dropout(0.25)(x)

    # Head
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(512, activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.5)(x, training=True)  # MC Dropout always ON
    x = tf.keras.layers.Dense(256, activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.5)(x, training=True)  # MC Dropout always ON

    out = tf.keras.layers.Dense(
        n_classes,
        activation='softmax',
        name="predictions"
    )(x)

    model = tf.keras.Model(inp, out, name="MRI_Classifier")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy',
                 tf.keras.metrics.AUC(name='auc'),
                 tf.keras.metrics.Precision(name='precision'),
                 tf.keras.metrics.Recall(name='recall')]
    )
    return model

cnn = build_cnn(n_classes)
cnn.summary()

Model: "MRI_Classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ image_input (InputLayer)        │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 128, 128, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 128, 128, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 128, 128, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 64, 64, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 32, 32, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 32, 32, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │             

 Total params: 852,261 (3.25 MB)

 Trainable params: 849,317 (3.24 MB)

 Non-trainable params: 2,944 (11.50 KB)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

# Class weights
class_weights     = compute_class_weight(
    'balanced', classes=np.unique(y), y=y)
class_weight_dict = dict(enumerate(class_weights))
print("Class weights:", class_weight_dict)

# ← NEW callbacks
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=20,              # ← very patient
        restore_best_weights=True,
        min_delta=0.005,
        verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=7,
        min_lr=1e-7,
        verbose=1),
    tf.keras.callbacks.ModelCheckpoint(
        '/content/best_mri_model.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1)
]

# ← Single phase training only (no fine tuning needed)
print("\n=== FULL TRAINING ===")
history = cnn.fit(
    X_train, y_train,
    validation_split=0.15,
    epochs=30,
    batch_size=32,
    callbacks=callbacks,
    class_weight=class_weight_dict,
    verbose=1
)

# ← Dummy history_ft so plotting block works
history_ft = type('obj', (object,), {
    'history': {
        'accuracy':     [],
        'val_accuracy': [],
        'loss':         [],
        'val_loss':     []
    }
})()

print("Training complete!")

Class weights: {0: np.float64(1.0), 1: np.float64(1.0), 2: np.float64(1.0), 3: np.float64(1.0), 4: np.float64(1.0)}

=== FULL TRAINING ===
Epoch 1/30
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.2858 - auc: 0.6194 - loss: 2.1897 - precision: 0.3189 - recall: 0.2249
Epoch 1: val_accuracy improved from None to 0.21667, saving model to /content/best_mri_model.keras

Epoch 1: finished saving model to /content/best_mri_model.keras
43/43 ━━━━━━━━━━━━━━━━━━━━ 232s 5s/step - accuracy: 0.3669 - auc: 0.6856 - loss: 1.9473 - precision: 0.4103 - recall: 0.3044 - val_accuracy: 0.2167 - val_auc: 0.4988 - val_loss: 2.1409 - val_precision: 0.2167 - val_recall: 0.2167 - learning_rate: 0.0010
Epoch 2/30
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.4681 - auc: 0.7679 - loss: 1.6301 - precision: 0.5041 - recall: 0.3909
Epoch 2: val_accuracy did not improve from 0.21667
43/43 ━━━━━━━━━━━━━━━━━━━━ 262s 5s/step - accuracy: 0.5132 - auc: 0.8013 - loss: 1.4399 - precision: 0.5551 - recall: 0.4331

In [ ]:

acc   = history.history['accuracy']    + history_ft.history['accuracy']
val_a = history.history['val_accuracy']+ history_ft.history['val_accuracy']
loss  = history.history['loss']        + history_ft.history['loss']
val_l = history.history['val_loss']    + history_ft.history['val_loss']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ep = range(1, len(acc)+1)

axes[0].plot(ep, acc,   'b-o', ms=4, label='Train Accuracy')
axes[0].plot(ep, val_a, 'r-s', ms=4, label='Val Accuracy')
axes[0].set_title("Accuracy over Epochs", fontweight='bold')
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(ep, loss,  'b-o', ms=4, label='Train Loss')
axes[1].plot(ep, val_l, 'r-s', ms=4, label='Val Loss')
axes[1].set_title("Loss over Epochs", fontweight='bold')
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Cross-Entropy Loss")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle("Training Dynamics", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/fig2_training_history.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: fig2_training_history.png")

Saved: fig2_training_history.png


In [ ]:
test_loss, test_acc, test_auc, test_prec, test_rec = cnn.evaluate(
    X_test, y_test, verbose=0)
print(f"\n{'='*50}")
print(f"  Test Accuracy : {test_acc:.4f}")
print(f"  Test AUC      : {test_auc:.4f}")
print(f"  Test Precision: {test_prec:.4f}")
print(f"  Test Recall   : {test_rec:.4f}")
print(f"  Test Loss     : {test_loss:.4f}")
print(f"{'='*50}\n")

preds  = cnn.predict(X_test, verbose=0)
y_pred = preds.argmax(axis=1)
y_true = y_test.argmax(axis=1)
print(classification_report(y_true, y_pred, target_names=classes))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(max(8, n_classes), max(6, n_classes-1)))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes, ax=ax,
            linewidths=0.5, linecolor='white')
ax.set_title("Confusion Matrix – Test Set", fontsize=13, fontweight='bold')
ax.set_xlabel("Predicted Label", fontsize=11)
ax.set_ylabel("True Label", fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('/content/fig3_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: fig3_confusion_matrix.png")


  Test Accuracy : 0.9500
  Test AUC      : 0.9952
  Test Precision: 0.9571
  Test Recall   : 0.9475
  Test Loss     : 0.1594

                                  precision    recall  f1-score   support

     fukuyama_muscular_dystrophy       0.90      0.93      0.91        80
      hallervorden_spatz_disease       1.00      0.86      0.93        80
                moyamoya_disease       0.91      1.00      0.95        80
pachygyria_cerebellar_hypoplasia       1.00      0.99      0.99        80
         walker_warburg_syndrome       0.95      0.97      0.96        80

                        accuracy                           0.95       400
                       macro avg       0.95      0.95      0.95       400
                    weighted avg       0.95      0.95      0.95       400

Saved: fig3_confusion_matrix.png


In [ ]:
def mc_dropout_predict(model, x, T=MC_SAMPLES):
    all_preds = []
    # Ensure dropout is active for each forward pass by explicitly passing training=True
    for _ in range(T):
        pred = model(x, training=True).numpy()
        all_preds.append(pred)
    preds = np.array(all_preds, dtype=np.float64)
    return preds.mean(axis=0), preds.var(axis=0)

print("Computing uncertainty on training set...")
train_unc, train_energy = [], []
for i in range(len(X_train)):
    mean, var = mc_dropout_predict(cnn, X_train[i:i+1])
    train_unc.append(float(var.mean()))

    logits = np.log(mean + 1e-10)
    train_energy.append(-logsumexp(logits))

train_unc    = np.array(train_unc)
train_energy = np.array(train_energy)

UNC_T = float(np.percentile(train_unc,    95))
OOD_T = float(np.percentile(train_energy, 95))
print(f"Uncertainty Threshold : {UNC_T:.6f}")
print(f"OOD Energy Threshold  : {OOD_T:.6f}")

print("Computing uncertainty on test set...")
test_unc, test_energy = [], []
for i in range(len(X_test)):
    mean, var = mc_dropout_predict(cnn, X_test[i:i+1])
    test_unc.append(float(var.mean()))
    # Ensure logits are also calculated with float64 precision
    logits = np.log(mean + 1e-10)
    test_energy.append(-logsumexp(logits))

test_unc    = np.array(test_unc)
test_energy = np.array(test_energy)
print(f"Avg Test Uncertainty : {test_unc.mean():.6f}")
print(f"Avg Test Energy Score: {test_energy.mean():.6f}")

Computing uncertainty on training set...
Uncertainty Threshold : 0.000072
OOD Energy Threshold  : 0.000000
Computing uncertainty on test set...
Avg Test Uncertainty : 0.000055
Avg Test Energy Score: 0.000000


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(train_unc, bins=40, alpha=0.6, color='steelblue', label='Train (In-dist.)')
axes[0].hist(test_unc,  bins=40, alpha=0.6, color='tomato',    label='Test')
axes[0].axvline(UNC_T, color='black', linestyle='--', linewidth=2,
                label=f'Threshold={UNC_T:.4f}')
axes[0].set_title("MC Dropout Uncertainty Distribution", fontweight='bold')
axes[0].set_xlabel("Predictive Variance (Uncertainty)")
axes[0].set_ylabel("Count")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].hist(train_energy, bins=40, alpha=0.6, color='steelblue', label='Train (In-dist.)')
axes[1].hist(test_energy,  bins=40, alpha=0.6, color='tomato',    label='Test')
axes[1].axvline(OOD_T, color='black', linestyle='--', linewidth=2,
                label=f'Threshold={OOD_T:.4f}')
axes[1].set_title("Energy Score Distribution (OOD Detection)", fontweight='bold')
axes[1].set_xlabel("Energy Score (−logsumexp)")
axes[1].set_ylabel("Count")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle("Uncertainty & OOD Score Distributions",
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/fig4_uncertainty_ood.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: fig4_uncertainty_ood.png")

Saved: fig4_uncertainty_ood.png


In [ ]:
def build_ae() -> tf.keras.Model:
    inp = tf.keras.Input((IMG_SIZE, IMG_SIZE, 3), name="ae_input")

    x       = tf.keras.layers.Conv2D(32,  3, activation='relu', padding='same')(inp)
    x       = tf.keras.layers.MaxPooling2D()(x)
    x       = tf.keras.layers.Conv2D(64,  3, activation='relu', padding='same')(x)
    x       = tf.keras.layers.MaxPooling2D()(x)
    encoded = tf.keras.layers.Conv2D(128, 3, activation='relu', padding='same')(x)

    x       = tf.keras.layers.Conv2DTranspose(64, 3, activation='relu', padding='same')(encoded)
    x       = tf.keras.layers.UpSampling2D()(x)
    x       = tf.keras.layers.Conv2DTranspose(32, 3, activation='relu', padding='same')(x)
    x       = tf.keras.layers.UpSampling2D()(x)
    decoded = tf.keras.layers.Conv2D(3, 3, activation='sigmoid',
                                     padding='same', name="reconstruction")(x)

    model = tf.keras.Model(inp, decoded, name="Autoencoder_OOD")
    model.compile(optimizer='adam', loss='mse')
    return model

ae = build_ae()
ae.fit(
    X_train, X_train,
    epochs=40,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    callbacks=[tf.keras.callbacks.EarlyStopping(
        patience=6, restore_best_weights=True)],
    verbose=1
)

train_recon_err = np.mean((ae.predict(X_train, verbose=0) - X_train)**2, axis=(1,2,3))
test_recon_err  = np.mean((ae.predict(X_test,  verbose=0) - X_test)**2,  axis=(1,2,3))
AE_OOD_T        = float(np.percentile(train_recon_err, 95))
print(f"Autoencoder OOD Threshold: {AE_OOD_T:.6f}")

Epoch 1/40
90/90 ━━━━━━━━━━━━━━━━━━━━ 116s 1s/step - loss: 0.0438 - val_loss: 0.0253
Epoch 2/40
90/90 ━━━━━━━━━━━━━━━━━━━━ 113s 1s/step - loss: 0.0235 - val_loss: 0.0207
Epoch 3/40
90/90 ━━━━━━━━━━━━━━━━━━━━ 113s 1s/step - loss: 0.0209 - val_loss: 0.0193
Epoch 4/40
90/90 ━━━━━━━━━━━━━━━━━━━━ 143s 1s/step - loss: 0.0183 - val_loss: 0.0166
Epoch 5/40
90/90 ━━━━━━━━━━━━━━━━━━━━ 111s 1s/step - loss: 0.0172 - val_loss: 0.0157
Epoch 6/40
90/90 ━━━━━━━━━━━━━━━━━━━━ 114s 1s/step - loss: 0.0162 - val_loss: 0.0148
Epoch 7/40
90/90 ━━━━━━━━━━━━━━━━━━━━ 141s 1s/step - loss: 0.0154 - val_loss: 0.0142
Epoch 8/40
90/90 ━━━━━━━━━━━━━━━━━━━━ 142s 1s/step - loss: 0.0151 - val_loss: 0.0138
Epoch 9/40
90/90 ━━━━━━━━━━━━━━━━━━━━ 113s 1s/step - loss: 0.0144 - val_loss: 0.0135
Epoch 10/40
90/90 ━━━━━━━━━━━━━━━━━━━━ 140s 1s/step - loss: 0.0143 - val_loss: 0.0132
Epoch 11/40
90/90 ━━━━━━━━━━━━━━━━━━━━ 142s 1s/step - loss: 0.0137 - val_loss: 0.0131
Epoch 12/40
90/90 ━━━━━━━━━━━━━━━━━━━━ 143s 1s/step - loss: 0.0

In [ ]:
def ece_score(probs, labels, n_bins=15):
    confidences = probs.max(axis=1)
    predictions = probs.argmax(axis=1)
    corrects    = (predictions == labels)

    bins     = np.linspace(0, 1, n_bins + 1)
    bin_acc  = np.zeros(n_bins)
    bin_conf = np.zeros(n_bins)
    bin_cnt  = np.zeros(n_bins)

    for i in range(n_bins):
        mask = (confidences > bins[i]) & (confidences <= bins[i+1])
        if mask.sum() > 0:
            bin_acc[i]  = corrects[mask].mean()
            bin_conf[i] = confidences[mask].mean()
            bin_cnt[i]  = mask.sum()

    ece = float(np.sum(bin_cnt * np.abs(bin_acc - bin_conf)) / len(probs))
    return ece, bin_acc, bin_conf, bin_cnt

ece, bin_acc, bin_conf, bin_cnt = ece_score(preds, y_true)
print(f"Expected Calibration Error (ECE): {ece:.4f}")

fig, ax = plt.subplots(figsize=(7, 6))
n_bins  = len(bin_acc)
bins    = np.linspace(0, 1, n_bins + 1)
bin_mid = (bins[:-1] + bins[1:]) / 2

ax.bar(bin_mid, bin_acc, width=1/n_bins, alpha=0.7,
       color='steelblue', edgecolor='black', label='Model Accuracy')
ax.plot([0,1], [0,1], 'r--', linewidth=2, label='Perfect Calibration')
ax.fill_between(bin_mid, bin_acc, bin_mid,
                alpha=0.2, color='tomato', label='Calibration Gap')
ax.set_title(f"Reliability Diagram  (ECE = {ece:.4f})",
             fontweight='bold', fontsize=12)
ax.set_xlabel("Confidence (max softmax)", fontsize=11)
ax.set_ylabel("Accuracy", fontsize=11)
ax.legend()
ax.grid(alpha=0.3)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('/content/fig5_reliability_diagram.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: fig5_reliability_diagram.png")

Expected Calibration Error (ECE): 0.0133
Saved: fig5_reliability_diagram.png


In [ ]:
def energy_score(logits):
    return float(-logsumexp(logits))

def decision(conf, unc, ood_energy, ae_err=None):
    flags = []
    if unc       > UNC_T:    flags.append("HIGH_UNCERTAINTY")
    if ood_energy > OOD_T:   flags.append("DISTRIBUTION_SHIFT")
    if ae_err is not None and ae_err > AE_OOD_T:
        flags.append("HIGH_RECONSTRUCTION_ERROR")

    label = "⚠️  Refer to Clinician" if flags else "✅ Model Decision (Reliable)"
    return {
        "label":      label,
        "confidence": round(conf, 4),
        "uncertainty":round(unc, 6),
        "energy":     round(ood_energy, 4),
        "ae_error":   round(ae_err, 6) if ae_err is not None else None,
        "flags":      flags
    }

print("\n--- Sample Predictions on Test Set ---")
for i in range(min(8, len(X_test))):
    mean, var = mc_dropout_predict(cnn, X_test[i:i+1])
    conf      = float(mean.max())
    unc       = float(var.mean())
    logits    = np.log(mean + 1e-10)
    ood       = energy_score(logits)
    ae_err    = float(np.mean((ae.predict(X_test[i:i+1], verbose=0)
                               - X_test[i:i+1])**2))
    result    = decision(conf, unc, ood, ae_err)
    true_cls  = classes[y_true[i]]
    pred_cls  = classes[mean.argmax()]
    print(f"[{i}] True={true_cls:30s} Pred={pred_cls:30s} "
          f"Conf={conf:.3f} {result['label']} Flags={result['flags']}")

In [ ]:
fig = plt.figure(figsize=(15, 6))
gs  = gridspec.GridSpec(1, 7, figure=fig, wspace=0.1)

boxes = [
    ("INPUT\nMRI Image\n128×128×3",           '#AED6F1'),
    ("PREPROCESS\nCLAHE + Z-norm\n3-channel float", '#A9DFBF'),
    ("EfficientNetB0\nBackbone\n(ImageNet)",   '#F9E79F'),
    ("MC Dropout\nHead\nT=20 passes",          '#F1948A'),
    ("Softmax\nPrediction\n+ Confidence",      '#D7BDE2'),
    ("UNCERTAINTY\nESTIMATION\nVar(preds)",    '#FAD7A0'),
    ("TRIAGE\nDECISION\nOOD / Refer",          '#ABEBC6'),
]

for j, (label, color) in enumerate(boxes):
    ax = fig.add_subplot(gs[j])
    ax.add_patch(plt.Rectangle((0.05, 0.1), 0.9, 0.8,
                                facecolor=color, edgecolor='black',
                                linewidth=1.5, transform=ax.transAxes,
                                clip_on=False))
    ax.text(0.5, 0.5, label, ha='center', va='center',
            fontsize=9, fontweight='bold', transform=ax.transAxes,
            multialignment='center')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.axis('off')
    if j < len(boxes) - 1:
        ax.annotate('', xy=(1.05, 0.5), xytext=(0.95, 0.5),
                    xycoords='axes fraction', textcoords='axes fraction',
                    arrowprops=dict(arrowstyle='->', color='black', lw=2))

fig.suptitle(
    "Proposed Framework: Uncertainty-Aware MRI Classification Pipeline",
    fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/content/fig6_pipeline_diagram.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: fig6_pipeline_diagram.png")

Saved: fig6_pipeline_diagram.png
